# Import

In [1]:
import pandas as pd
import json

from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[
        logging.FileHandler("curation.log"),
        logging.StreamHandler(),  # keep console output too
    ],
    force=True,
)

# Download data

In [2]:
noncurated_path = "../non_curated/h5ad/norman_2019_raw.h5ad"
download_file(
    url="https://exampledata.scverse.org/pertpy/norman_2019_raw.h5ad",
    dest_path=noncurated_path
)

File ../non_curated/h5ad/norman_2019_raw.h5ad already exists. Skipping download.


# Initialise the dataset object

In [18]:
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    exp_metadata_schema=Experiment,
    noncurated_path=noncurated_path
)

cur_data.load_data()

Loading data from ../non_curated/h5ad/norman_2019_raw.h5ad


In [19]:
cur_data.adata.obs

,guide_identity,read_count,UMI_count,coverage,gemgroup,good_coverage,number_of_cells,guide_AHR,guide_ARID1A,guide_ARRDC3,...,guide_TP73,guide_TSC22D1,guide_UBASH3A,guide_UBASH3B,guide_ZBTB1,guide_ZBTB10,guide_ZBTB25,guide_ZC3HAV1,guide_ZNF318,guide_ids
index,,,,,,,,,,,,,,,,,,,,,
AAACCTGAGAAGAAGC-1,NegCtrl0_NegCtrl0__NegCtrl0_NegCtrl0,1252,67,18.686567,1,True,2,0,0,0,...,0,0,0,0,0,0,0,0,0,
AAACCTGAGGCATGTG-1,TSC22D1_NegCtrl0__TSC22D1_NegCtrl0,2151,104,20.682692,1,True,1,0,0,0,...,0,1,0,0,0,0,0,0,0,TSC22D1
AAACCTGAGGCCCTTG-1,KLF1_MAP2K6__KLF1_MAP2K6,1037,59,17.576271,1,True,1,0,0,0,...,0,0,0,0,0,0,0,0,0,"KLF1,MAP2K6"
AAACCTGCACGAAGCA-1,NegCtrl10_NegCtrl0__NegCtrl10_NegCtrl0,958,39,24.564103,1,True,1,0,0,0,...,0,0,0,0,0,0,0,0,0,
AAACCTGCAGACGTAG-1,CEBPE_RUNX1T1__CEBPE_RUNX1T1,244,14,17.428571,1,True,1,0,0,0,...,0,0,0,0,0,0,0,0,0,"CEBPE,RUNX1T1"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCATCAGTACGT-8,FOXA3_NegCtrl0__FOXA3_NegCtrl0,2068,95,21.768421,8,True,1,0,0,0,...,0,0,0,0,0,0,0,0,0,FOXA3
TTTGTCATCCACTCCA-8,CELF2_NegCtrl0__CELF2_NegCtrl0,829,33,25.121212,8,True,1,0,0,0,...,0,0,0,0,0,0,0,0,0,CELF2
TTTGTCATCCCAACGG-8,BCORL1_NegCtrl0__BCORL1_NegCtrl0,136,9,15.111111,8,True,1,0,0,0,...,0,0,0,0,0,0,0,0,0,BCORL1


# OBS slot curation

### Drop all columns starting with `guide_`

In [20]:
cur_data.adata.obs = cur_data.adata.obs[['guide_identity', 'guide_ids', 'gemgroup']]

### Add cell barcodes to the obs slot

In [21]:
cur_data.adata.obs['cell_barcode'] = cur_data.adata.obs.index.astype(str)

print(cur_data.adata.obs[['cell_barcode']].head())

                          cell_barcode
index                                 
AAACCTGAGAAGAAGC-1  AAACCTGAGAAGAAGC-1
AAACCTGAGGCATGTG-1  AAACCTGAGGCATGTG-1
AAACCTGAGGCCCTTG-1  AAACCTGAGGCCCTTG-1
AAACCTGCACGAAGCA-1  AAACCTGCACGAAGCA-1
AAACCTGCAGACGTAG-1  AAACCTGCAGACGTAG-1


### Show unique perturbations

In [22]:
cur_data.show_unique(slot = 'obs', column = 'guide_identity')

Unique values in adata.obs.guide_identity: 290
--------------------------------------------------
{'AHR_FEV__AHR_FEV',
 'AHR_KLF1__AHR_KLF1',
 'AHR_NegCtrl0__AHR_NegCtrl0',
 'ARID1A_NegCtrl0__ARID1A_NegCtrl0',
 'ARRDC3_NegCtrl0__ARRDC3_NegCtrl0',
 'ATL1_NegCtrl0__ATL1_NegCtrl0',
 'BAK1_NegCtrl0__BAK1_NegCtrl0',
 'BCL2L11_BAK1__BCL2L11_BAK1',
 'BCL2L11_NegCtrl0__BCL2L11_NegCtrl0',
 'BCL2L11_TGFBR2__BCL2L11_TGFBR2',
 'BCORL1_NegCtrl0__BCORL1_NegCtrl0',
 'BPGM_NegCtrl0__BPGM_NegCtrl0',
 'BPGM_SAMD1__BPGM_SAMD1',
 'BPGM_ZBTB1__BPGM_ZBTB1',
 'C19orf26_NegCtrl0__C19orf26_NegCtrl0',
 'C3orf72_FOXL2__C3orf72_FOXL2',
 'C3orf72_NegCtrl0__C3orf72_NegCtrl0',
 'CBFA2T3_NegCtrl0__CBFA2T3_NegCtrl0',
 'CBL_CNN1__CBL_CNN1',
 'CBL_NegCtrl0__CBL_NegCtrl0',
 'CBL_PTPN12__CBL_PTPN12',
 'CBL_PTPN9__CBL_PTPN9',
 'CBL_TGFBR2__CBL_TGFBR2',
 'CBL_UBASH3A__CBL_UBASH3A',
 'CBL_UBASH3B__CBL_UBASH3B',
 'CDKN1A_NegCtrl0__CDKN1A_NegCtrl0',
 'CDKN1B_CDKN1A__CDKN1B_CDKN1A',
 'CDKN1B_NegCtrl0__CDKN1B_NegCtrl0',
 'CDKN1C

### Rename `guide_identity` to `perturbation_name`

In [23]:
cur_data.rename_columns(slot = 'obs', name_dict = {'guide_identity': 'perturbation_name'})

Renamed columns in adata.obs: {'guide_identity': 'perturbation_name'}


In [24]:
# split perturbation_name to keep the first part only
cur_data.adata.obs['perturbation_name'] = cur_data.adata.obs['perturbation_name'].str.split('__').str[0]
cur_data.adata.obs[['perturbation_name']]

,perturbation_name
index,
AAACCTGAGAAGAAGC-1,NegCtrl0_NegCtrl0
AAACCTGAGGCATGTG-1,TSC22D1_NegCtrl0
AAACCTGAGGCCCTTG-1,KLF1_MAP2K6
AAACCTGCACGAAGCA-1,NegCtrl10_NegCtrl0
AAACCTGCAGACGTAG-1,CEBPE_RUNX1T1
...,...
TTTGTCATCAGTACGT-8,FOXA3_NegCtrl0
TTTGTCATCCACTCCA-8,CELF2_NegCtrl0
TTTGTCATCCCAACGG-8,BCORL1_NegCtrl0


In [25]:
cur_data.adata.obs

,perturbation_name,guide_ids,gemgroup,cell_barcode
index,,,,
AAACCTGAGAAGAAGC-1,NegCtrl0_NegCtrl0,,1,AAACCTGAGAAGAAGC-1
AAACCTGAGGCATGTG-1,TSC22D1_NegCtrl0,TSC22D1,1,AAACCTGAGGCATGTG-1
AAACCTGAGGCCCTTG-1,KLF1_MAP2K6,"KLF1,MAP2K6",1,AAACCTGAGGCCCTTG-1
AAACCTGCACGAAGCA-1,NegCtrl10_NegCtrl0,,1,AAACCTGCACGAAGCA-1
AAACCTGCAGACGTAG-1,CEBPE_RUNX1T1,"CEBPE,RUNX1T1",1,AAACCTGCAGACGTAG-1
...,...,...,...,...
TTTGTCATCAGTACGT-8,FOXA3_NegCtrl0,FOXA3,8,TTTGTCATCAGTACGT-8
TTTGTCATCCACTCCA-8,CELF2_NegCtrl0,CELF2,8,TTTGTCATCCACTCCA-8
TTTGTCATCCCAACGG-8,BCORL1_NegCtrl0,BCORL1,8,TTTGTCATCCCAACGG-8


### Add guide RNA information

In [26]:
# download the guide RNA spreadsheet
download_file(
    url="http://www.science.org/doi/suppl/10.1126/science.aax4438/suppl_file/aax4438_tables2.xlsx",
    dest_path="../supplementary/norman_2019_guide_info.xlsx"
)

# read in the guide RNA spreadsheet
# guides for the K562 essential day 6 library are in "TabB_K562_day6_library"
guide_info_df = pd.read_excel("../supplementary/norman_2019_guide_info.xlsx", sheet_name="Perturbseq_sgRNA_info")

# create perturbation_name column in guide_info_df
guide_info_df['perturbation_name'] = guide_info_df['gene_A'] + '_' + guide_info_df['gene_B']
# check that all perturbation names in cur_data are in guide_info_df
print(f"All perturbation names in cur_data are in guide_info_df: {cur_data.adata.obs['perturbation_name'].isin(guide_info_df['perturbation_name']).all()}")
# create guide_sequence column in guide_info_df
guide_info_df['guide_sequence'] = guide_info_df['protospacer_sequence_A'] + '|' + guide_info_df['protospacer_sequence_B']
# subset for necessary columns
guide_info_df = guide_info_df[['perturbation_name', 'guide_sequence']]
# merge cur_data.adata.obs with guide_info_df on perturbation_name
cur_data.adata.obs = cur_data.adata.obs.merge(guide_info_df, on='perturbation_name', how='left')
# # check that there are no missing guide sequences
print(f"Number of missing guide sequences: {cur_data.adata.obs['guide_sequence'].isna().sum()}")

File ../supplementary/norman_2019_guide_info.xlsx already exists. Skipping download.
All perturbation names in cur_data are in guide_info_df: True
Number of missing guide sequences: 0


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Standardise perturbation targets

In [27]:
cur_data.adata.obs['target'] = cur_data.adata.obs['perturbation_name'].str.replace('_', '|').str.replace(r'NegCtrl\d+', 'control_nontargeting', regex=True)
cur_data.adata.obs[['target']]

,target
0,control_nontargeting|control_nontargeting
1,TSC22D1|control_nontargeting
2,KLF1|MAP2K6
3,control_nontargeting|control_nontargeting
4,CEBPE|RUNX1T1
...,...
111440,FOXA3|control_nontargeting
111441,CELF2|control_nontargeting
111442,BCORL1|control_nontargeting
111443,ZBTB10|PTPN12


In [28]:
cur_data.standardize_genes(
    slot='obs',
    input_column='target',
    input_column_type='gene_symbol',
    multiple_entries=True,
    multiple_entries_sep='|'
)

Mapping gene symbols: 100%|████████████████████████████████████| 106/106 [00:00<00:00, 25409.85it/s]


--------------------------------------------------
Successfully mapped 106 out of 106 gene symbols.
--------------------------------------------------
Couldn't map gene symbols: []
--------------------------------------------------
Collapsed column positional_index using separator |


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Add `perturbed_target_number` column

In [29]:
cur_data.count_entries(
    slot='obs',
    input_column='perturbed_target_symbol',
    count_column_name='perturbed_target_number',
    sep='|'
)

Counted entries in column perturbed_target_symbol of adata.obs and stored in perturbed_target_number


### Encode chromosomes as integers

In [30]:
cur_data.chromosome_encoding()

Chromosome encoding applied to perturbed_target_chromosome in adata.obs and stored as 'perturbed_target_chromosome_encoding'.


### Assign replicates

In [31]:
cur_data.adata.obs = cur_data.adata.obs.rename(columns={'gemgroup': 'technical_replicate'})

In [32]:
cur_data.adata.obs

,target,perturbation_name,guide_sequence,guide_ids,technical_replicate,cell_barcode,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,perturbed_target_chromosome,original_index,perturbed_target_number,perturbed_target_chromosome_encoding
index,,,,,,,,,,,,,,
0,control_nontargeting|control_nontargeting,NegCtrl0_NegCtrl0,GTCGCGCCCGCTCCAGGGAC|GTCGCGCCCGCTCCAGGGAC,,1,AAACCTGAGAAGAAGC-1,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,0|0,1,0
1,TSC22D1|control_nontargeting,TSC22D1_NegCtrl0,GTCCCCGGAGCTGTGTCCCG|GTCGCGCCCGCTCCAGGGAC,TSC22D1,1,AAACCTGAGGCATGTG-1,ENSG00000102804|control_nontargeting,TSC22D1|control_nontargeting,protein_coding|control_nontargeting,chr13:44432143-44577316;-1|control_nontargeting,13|control_nontargeting,1|1,2,0
2,KLF1|MAP2K6,KLF1_MAP2K6,GGGGCTGTGGAGCCTCAATC|GGTTCTCCGGCGGAGTCCAC,"KLF1,MAP2K6",1,AAACCTGAGGCCCTTG-1,ENSG00000105610|ENSG00000108984,KLF1|MAP2K6,protein_coding|protein_coding,chr19:12884422-12887201;-1|chr17:69414683-6955...,19|17,2|2,2,0
3,control_nontargeting|control_nontargeting,NegCtrl10_NegCtrl0,GCTCGCGTCTGCCGCCAGAC|GTCGCGCCCGCTCCAGGGAC,,1,AAACCTGCACGAAGCA-1,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,3|3,1,0
4,CEBPE|RUNX1T1,CEBPE_RUNX1T1,GCCCCTCAAAAAACAAACCC|GCGCCGGAGCCGGCCTGATG,"CEBPE,RUNX1T1",1,AAACCTGCAGACGTAG-1,ENSG00000092067|ENSG00000079102,CEBPE|RUNX1T1,protein_coding|protein_coding,chr14:23117036-23120256;-1|chr8:91954972-92103...,14|8,4|4,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111440,FOXA3|control_nontargeting,FOXA3_NegCtrl0,GAGACAGACCTCAGGCGCCG|GTCGCGCCCGCTCCAGGGAC,FOXA3,8,TTTGTCATCAGTACGT-8,ENSG00000170608|control_nontargeting,FOXA3|control_nontargeting,protein_coding|control_nontargeting,chr19:45863989-45873798;1|control_nontargeting,19|control_nontargeting,111440|111440,2,0
111441,CELF2|control_nontargeting,CELF2_NegCtrl0,GCTCCGCCCCGCGGGGACAC|GTCGCGCCCGCTCCAGGGAC,CELF2,8,TTTGTCATCCACTCCA-8,ENSG00000048740|control_nontargeting,CELF2|control_nontargeting,protein_coding|control_nontargeting,chr10:10798397-11336675;1|control_nontargeting,10|control_nontargeting,111441|111441,2,0
111442,BCORL1|control_nontargeting,BCORL1_NegCtrl0,GGATCGCTGAGAGGACCGAG|GTCGCGCCCGCTCCAGGGAC,BCORL1,8,TTTGTCATCCCAACGG-8,ENSG00000085185|control_nontargeting,BCORL1|control_nontargeting,protein_coding|control_nontargeting,chrX:129981107-130058071;1|control_nontargeting,X|control_nontargeting,111442|111442,2,0


### Add metadata

In [33]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        #----- dataset -----#
        "dataset_id": cur_data.dataset_id,
        #----- sample -----#
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        #----- perturbation type -----#
        "perturbation_type_label": "CRISPRa",
        "perturbation_type_id": None,
        #----- data modality -----#
        "data_modality": "Perturb-seq", # different from "method_name_label"; more general term - choice of CRISPR, MAVE and Perturb-seq
        #----- significance -----#
        "significant": None,
        "significance_criteria": None,
        #----- score interpretation -----#
        "score_interpretation": None,
        #----- treatment -----#
        "treatment_label": None,
        "treatment_id": None,
        #----- replicate -----#
        # "technical_replicate": None,
        "biological_replicate": None,
        #----- model system -----#
        "model_system_label": "cell_line",
        "model_system_id": None,
        #----- tissue -----#
        "tissue": "blood",
        #----- cell line -----#
        "cell_line_label": "K 562 cell",
        "cell_line_id": None,
        #----- cell type -----#
        "cell_type_label": "lymphoblast",
        "cell_type_id": None,
        #----- disease -----#
        "disease_label": "chronic myelogenous leukemia, BCR-ABL1 positive",
        "disease_id": None,
        #----- timepoint -----#
        "timepoint": "P7DT0H0M0S",
        #----- species -----#
        "species": "Homo sapiens",
        #----- sex -----#
        "sex_label": "female",
        "sex_id": None,
        #----- developmental stage -----#
        "developmental_stage_label": "adult",
        "developmental_stage_id": None,
        #----- study metadata -----#
        "study_title": "Exploring genetic interaction manifolds constructed from rich single-cell phenotypes",
        "study_uri": "https://doi.org/10.1126/science.aax4438",
        "study_year": 2019,
        #----- authors -----#
        "first_author": "Thomas M. Norman",
        "last_author": "Jonathan S. Weissman",
        #----- experiment metadata -----#
        "experiment_title": "Perturb-seq CRISPRa of K562 cells to explore genetic interaction manifolds",
        "experiment_summary": """
            K562 cells were engineered by lentiviral transduction of dox-inducible CRISPRa SunTag system. The sgRNA library consisting of 295 dual sgRNA vectors was packaged into lentiviral particles, spinfected into K562 cells. Cells were grown until day 7, at which point they were harvested, processed using Chromium Single Cell 3-prime Gel Beads v2 kit and sequenced on an Illumina NovaSeq 6000.
        """,
        #----- number of perturbed targets/samples -----#
        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_coord'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],
        #----- library generation type -----#
        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",
        #----- library generation method -----#
        "library_generation_method_id": "EFO:0022898",
        "library_generation_method_label": "dCas9-Suntag",
        #----- enzyme and library delivery method -----#
        "enzyme_delivery_method_id": None,
        "enzyme_delivery_method_label": "lentivirus transduction",

        "library_delivery_method_id": None,
        "library_delivery_method_label": "lentivirus transduction",
        #----- enzyme and library integration state -----#
        "enzyme_integration_state_id": None,
        "enzyme_integration_state_label": "random locus integration",

        "library_integration_state_id": None,
        "library_integration_state_label": "random locus integration",
        #----- enzyme and library expression control -----#
        "enzyme_expression_control_id": None,
        "enzyme_expression_control_label": "inducible transgene expression",

        "library_expression_control_id": None,
        "library_expression_control_label": "constitutive transgene expression",
        #----- library name and URI and manufacturer -----#
        "library_name": "custom",
        "library_uri": None,
        "library_manufacturer": "Weissman lab",
        #----- library format -----#
        "library_format_id": None,
        "library_format_label": "pooled",
        #----- library scope -----#
        "library_scope_id": None,
        "library_scope_label": "focused",
        #----- library perturbation type -----#
        "library_perturbation_type_id": None,
        "library_perturbation_type_label": "activation",
        #----- library additional metadata -----#
        "library_lentiviral_generation": "3",
        "library_grnas_per_target": "1",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()), # for CRISPR/Perturb-seq
        "library_total_variants": None, # for MAVE
        #----- readout dimensionality -----#
        "readout_dimensionality_id": None,
        "readout_dimensionality_label": "high-dimensional assay",
        #---- readout type -----#
        "readout_type_id": None,
        "readout_type_label": "transcriptomic",
        #----- readout technology -----#
        "readout_technology_id": None,
        "readout_technology_label": "single-cell rna-seq",
        #----- method -----#
        "method_name_id": None,
        "method_name_label": "Perturb-seq", # different from "data_modality"; more specific term - specific name of the technique
        "method_uri": None,
        #----- sequencing library kit -----#
        "sequencing_library_kit_id": None,
        "sequencing_library_kit_label": "10x Genomics Single Cell 3-prime v2",
        #----- sequencing platform -----#
        "sequencing_platform_id": None,
        "sequencing_platform_label": "Illumina NovaSeq 6000",
        #----- sequencing strategy -----#
        "sequencing_strategy_id": None,
        "sequencing_strategy_label": "barcode sequencing",
        #----- software used for counts-----#
        "software_counts_id": None,
        "software_counts_label": "CellRanger",
        #----- software used for analysis -----#
        "software_analysis_id": None,
        "software_analysis_label": "custom",
        #----- reference genome -----#
        "reference_genome_id": None,
        "reference_genome_label": "GRCh38",
        #----- license -----#
        "license_label": "free to use license",
        "license_id": "SWO:1000061",
        #----- external datasets -----#
        "associated_datasets": json.dumps([
            {
                "dataset_accession": "GSE133344",
                "dataset_uri": "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE133344",
                "dataset_description": "Raw counts; matrix.mtx, features.tsv, barcodes.tsv",
                "dataset_file_name": "GSE278572_*.*",
            },
            {
                "dataset_accession": "norman_2019_raw.h5ad",
                "dataset_uri": "https://exampledata.scverse.org/pertpy/norman_2019_raw.h5ad",
                "dataset_description": "Raw counts - .h5ad file from pertpy",
                "dataset_file_name": "norman_2019_raw.h5ad",
            }
        ])
    }
)

Column dataset_id added to adata.obs
Column sample_id added to adata.obs
Column perturbation_type_label added to adata.obs
Column perturbation_type_id added to adata.obs
Column data_modality added to adata.obs
Column significant added to adata.obs
Column significance_criteria added to adata.obs
Column score_interpretation added to adata.obs
Column treatment_label added to adata.obs
Column treatment_id added to adata.obs
Column biological_replicate added to adata.obs
Column model_system_label added to adata.obs
Column model_system_id added to adata.obs
Column tissue added to adata.obs
Column cell_line_label added to adata.obs
Column cell_line_id added to adata.obs
Column cell_type_label added to adata.obs
Column cell_type_id added to adata.obs
Column disease_label added to adata.obs
Column disease_id added to adata.obs
Column timepoint added to adata.obs
Column species added to adata.obs
Column sex_label added to adata.obs
Column sex_id added to adata.obs
Column developmental_stage_labe

In [34]:
cur_data.adata.obs

,target,perturbation_name,guide_sequence,guide_ids,technical_replicate,cell_barcode,perturbed_target_ensg,perturbed_target_symbol,perturbed_target_biotype,perturbed_target_coord,...,sequencing_strategy_label,software_counts_id,software_counts_label,software_analysis_id,software_analysis_label,reference_genome_id,reference_genome_label,license_label,license_id,associated_datasets
index,,,,,,,,,,,,,,,,,,,,,
0,control_nontargeting|control_nontargeting,NegCtrl0_NegCtrl0,GTCGCGCCCGCTCCAGGGAC|GTCGCGCCCGCTCCAGGGAC,,1,AAACCTGAGAAGAAGC-1,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,...,barcode sequencing,None,CellRanger,None,custom,None,GRCh38,free to use license,SWO:1000061,"[{""dataset_accession"": ""GSE133344"", ""dataset_u..."
1,TSC22D1|control_nontargeting,TSC22D1_NegCtrl0,GTCCCCGGAGCTGTGTCCCG|GTCGCGCCCGCTCCAGGGAC,TSC22D1,1,AAACCTGAGGCATGTG-1,ENSG00000102804|control_nontargeting,TSC22D1|control_nontargeting,protein_coding|control_nontargeting,chr13:44432143-44577316;-1|control_nontargeting,...,barcode sequencing,None,CellRanger,None,custom,None,GRCh38,free to use license,SWO:1000061,"[{""dataset_accession"": ""GSE133344"", ""dataset_u..."
2,KLF1|MAP2K6,KLF1_MAP2K6,GGGGCTGTGGAGCCTCAATC|GGTTCTCCGGCGGAGTCCAC,"KLF1,MAP2K6",1,AAACCTGAGGCCCTTG-1,ENSG00000105610|ENSG00000108984,KLF1|MAP2K6,protein_coding|protein_coding,chr19:12884422-12887201;-1|chr17:69414683-6955...,...,barcode sequencing,None,CellRanger,None,custom,None,GRCh38,free to use license,SWO:1000061,"[{""dataset_accession"": ""GSE133344"", ""dataset_u..."
3,control_nontargeting|control_nontargeting,NegCtrl10_NegCtrl0,GCTCGCGTCTGCCGCCAGAC|GTCGCGCCCGCTCCAGGGAC,,1,AAACCTGCACGAAGCA-1,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,...,barcode sequencing,None,CellRanger,None,custom,None,GRCh38,free to use license,SWO:1000061,"[{""dataset_accession"": ""GSE133344"", ""dataset_u..."
4,CEBPE|RUNX1T1,CEBPE_RUNX1T1,GCCCCTCAAAAAACAAACCC|GCGCCGGAGCCGGCCTGATG,"CEBPE,RUNX1T1",1,AAACCTGCAGACGTAG-1,ENSG00000092067|ENSG00000079102,CEBPE|RUNX1T1,protein_coding|protein_coding,chr14:23117036-23120256;-1|chr8:91954972-92103...,...,barcode sequencing,None,CellRanger,None,custom,None,GRCh38,free to use license,SWO:1000061,"[{""dataset_accession"": ""GSE133344"", ""dataset_u..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111440,FOXA3|control_nontargeting,FOXA3_NegCtrl0,GAGACAGACCTCAGGCGCCG|GTCGCGCCCGCTCCAGGGAC,FOXA3,8,TTTGTCATCAGTACGT-8,ENSG00000170608|control_nontargeting,FOXA3|control_nontargeting,protein_coding|control_nontargeting,chr19:45863989-45873798;1|control_nontargeting,...,barcode sequencing,None,CellRanger,None,custom,None,GRCh38,free to use license,SWO:1000061,"[{""dataset_accession"": ""GSE133344"", ""dataset_u..."
111441,CELF2|control_nontargeting,CELF2_NegCtrl0,GCTCCGCCCCGCGGGGACAC|GTCGCGCCCGCTCCAGGGAC,CELF2,8,TTTGTCATCCACTCCA-8,ENSG00000048740|control_nontargeting,CELF2|control_nontargeting,protein_coding|control_nontargeting,chr10:10798397-11336675;1|control_nontargeting,...,barcode sequencing,None,CellRanger,None,custom,None,GRCh38,free to use license,SWO:1000061,"[{""dataset_accession"": ""GSE133344"", ""dataset_u..."
111442,BCORL1|control_nontargeting,BCORL1_NegCtrl0,GGATCGCTGAGAGGACCGAG|GTCGCGCCCGCTCCAGGGAC,BCORL1,8,TTTGTCATCCCAACGG-8,ENSG00000085185|control_nontargeting,BCORL1|control_nontargeting,protein_coding|control_nontargeting,chrX:129981107-130058071;1|control_nontargeting,...,barcode sequencing,None,CellRanger,None,custom,None,GRCh38,free to use license,SWO:1000061,"[{""dataset_accession"": ""GSE133344"", ""dataset_u..."


### Curate tissue information


In [35]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

Mapped 1 tissue ontology terms from `tissue` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
  input_column input_column_lower name_lower     ontology_id
0        blood              blood      blood  UBERON:0000178
--------------------------------------------------


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate cell type information

In [36]:
cur_data.standardize_ontology(
    input_column='cell_type_label',
    column_type='term_name',
    ontology_type='cell_type',
    overwrite=True
)

Mapped 1 cell_type ontology terms from `cell_type_label` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
  input_column input_column_lower   name_lower ontology_id
0  lymphoblast        lymphoblast  lymphoblast  CL:0017005
--------------------------------------------------
Overwriting column cell_type_label in adata.obs
Overwriting column cell_type_id in adata.obs


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate cell line information

In [37]:
cur_data.standardize_ontology(
    input_column='cell_line_label',
    column_type='term_name',
    ontology_type='cell_line',
    overwrite=True
)

Mapped 1 cell_line ontology terms from `cell_line_label` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
  input_column input_column_lower  name_lower  ontology_id
0   K 562 cell         k 562 cell  k 562 cell  CLO:0007050
--------------------------------------------------
Overwriting column cell_line_label in adata.obs
Overwriting column cell_line_id in adata.obs


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Curate disease information

In [38]:
cur_data.standardize_ontology(
    input_column='disease_label',
    column_type='term_name',
    ontology_type='disease',
    overwrite=True
)

Mapped 1 disease ontology terms from `disease_label` column to ontology terms
DataFrame shape: (1, 4)
--------------------------------------------------
                                      input_column  \
0  chronic myelogenous leukemia, BCR-ABL1 positive   

                                input_column_lower  \
0  chronic myelogenous leukemia, bcr-abl1 positive   

                                        name_lower    ontology_id  
0  chronic myelogenous leukemia, bcr-abl1 positive  MONDO:0011996  
--------------------------------------------------
Overwriting column disease_label in adata.obs
Overwriting column disease_id in adata.obs


/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Match schema column order

In [39]:
cur_data.match_schema_columns(slot='obs')

Matched columns of adata.obs to the obs_schema.


### Validate obs metadata

In [40]:
cur_data.validate_data(slot='obs', verbose=True)

2026-04-28 16:37:04,856 INFO curation_tools.curation_tools: adata.obs is valid according to the obs_schema.


,dataset_id,sample_id,cell_barcode,data_modality,significant,significance_criteria,perturbation_name,perturbed_target_coord,perturbed_target_chromosome,perturbed_target_chromosome_encoding,...,software_counts_id,software_counts_label,software_analysis_id,software_analysis_label,score_interpretation,reference_genome_id,reference_genome_label,associated_datasets,license_label,license_id
0,norman_2019_raw,1,AAACCTGAGAAGAAGC-1,Perturb-seq,<NA>,<NA>,NegCtrl0_NegCtrl0,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,0,...,<NA>,CellRanger,<NA>,custom,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE133344"", ""dataset_u...",free to use license,SWO:1000061
1,norman_2019_raw,2,AAACCTGAGGCATGTG-1,Perturb-seq,<NA>,<NA>,TSC22D1_NegCtrl0,chr13:44432143-44577316;-1|control_nontargeting,13|control_nontargeting,0,...,<NA>,CellRanger,<NA>,custom,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE133344"", ""dataset_u...",free to use license,SWO:1000061
2,norman_2019_raw,3,AAACCTGAGGCCCTTG-1,Perturb-seq,<NA>,<NA>,KLF1_MAP2K6,chr19:12884422-12887201;-1|chr17:69414683-6955...,19|17,0,...,<NA>,CellRanger,<NA>,custom,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE133344"", ""dataset_u...",free to use license,SWO:1000061
3,norman_2019_raw,4,AAACCTGCACGAAGCA-1,Perturb-seq,<NA>,<NA>,NegCtrl10_NegCtrl0,control_nontargeting|control_nontargeting,control_nontargeting|control_nontargeting,0,...,<NA>,CellRanger,<NA>,custom,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE133344"", ""dataset_u...",free to use license,SWO:1000061
4,norman_2019_raw,5,AAACCTGCAGACGTAG-1,Perturb-seq,<NA>,<NA>,CEBPE_RUNX1T1,chr14:23117036-23120256;-1|chr8:91954972-92103...,14|8,0,...,<NA>,CellRanger,<NA>,custom,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE133344"", ""dataset_u...",free to use license,SWO:1000061
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111440,norman_2019_raw,111441,TTTGTCATCAGTACGT-8,Perturb-seq,<NA>,<NA>,FOXA3_NegCtrl0,chr19:45863989-45873798;1|control_nontargeting,19|control_nontargeting,0,...,<NA>,CellRanger,<NA>,custom,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE133344"", ""dataset_u...",free to use license,SWO:1000061
111441,norman_2019_raw,111442,TTTGTCATCCACTCCA-8,Perturb-seq,<NA>,<NA>,CELF2_NegCtrl0,chr10:10798397-11336675;1|control_nontargeting,10|control_nontargeting,0,...,<NA>,CellRanger,<NA>,custom,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE133344"", ""dataset_u...",free to use license,SWO:1000061
111442,norman_2019_raw,111443,TTTGTCATCCCAACGG-8,Perturb-seq,<NA>,<NA>,BCORL1_NegCtrl0,chrX:129981107-130058071;1|control_nontargeting,X|control_nontargeting,0,...,<NA>,CellRanger,<NA>,custom,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE133344"", ""dataset_u...",free to use license,SWO:1000061
111443,norman_2019_raw,111444,TTTGTCATCCTCCTAG-8,Perturb-seq,<NA>,<NA>,ZBTB10_PTPN12,chr8:80485596-80526265;1|chr7:77537295-77640072;1,8|7,0,...,<NA>,CellRanger,<NA>,custom,<NA>,<NA>,GRCh38,"[{""dataset_accession"": ""GSE133344"", ""dataset_u...",free to use license,SWO:1000061


# VAR slot curation

### Standardise genes

In [46]:
cur_data.adata.var

,gene_symbols,gene_ensembl_id,ensembl_gene_id,gene_symbol,original_index
index,,,,,
0,RP11-34P13.3,ENSG00000243485,ENSG00000243485,MIR1302-2HG,ENSG00000243485
1,FAM138A,ENSG00000237613,ENSG00000237613,FAM138A,ENSG00000237613
2,OR4F5,ENSG00000186092,ENSG00000186092,OR4F5,ENSG00000186092
3,RP11-34P13.7,ENSG00000238009,ENSG00000241860,RP11-34P13.7,ENSG00000238009
4,RP11-34P13.8,ENSG00000239945,ENSG00000239945,RP11-34P13.8,ENSG00000239945
...,...,...,...,...,...
33689,AC233755.2,ENSG00000277856,ENSG00000277856,AC233755.2,ENSG00000277856
33690,AC233755.1,ENSG00000275063,ENSG00000275063,AC233755.1,ENSG00000275063
33691,AC240274.1,ENSG00000271254,ENSG00000271254,AC240274.1,ENSG00000271254


In [42]:
cur_data.create_columns(
    slot = 'var',
    col_dict={'gene_ensembl_id': cur_data.adata.var.index},
    overwrite=True
)

Column gene_ensembl_id added to adata.var


In [43]:
cur_data.standardize_genes(
    slot="var",
    input_column="gene_ensembl_id",
    input_column_type="ensembl_gene_id",
    remove_version=False,
    multiple_entries=False
)

Missing Ensembl IDs: ['ENSG00000203818', 'ENSG00000223379', 'ENSG00000223797', 'ENSG00000254651', 'ENSG00000260517', 'ENSG00000270394', 'ENSG00000281657', 'ENSG00000278266', 'ENSG00000282031', 'ENSG00000229245', 'ENSG00000269051', 'ENSG00000253872', 'ENSG00000237359', 'ENSG00000261068', 'ENSG00000227869', 'ENSG00000232698', 'ENSG00000258407', 'ENSG00000248265', 'ENSG00000231846', 'ENSG00000273237', 'ENSG00000264790', 'ENSG00000237262', 'ENSG00000267667', 'ENSG00000282111', 'ENSG00000278674', 'ENSG00000242924', 'ENSG00000279131', 'ENSG00000269920', 'ENSG00000274762', 'ENSG00000272049', 'ENSG00000272922', 'ENSG00000274312', 'ENSG00000261350', 'ENSG00000249604', 'ENSG00000250410', 'ENSG00000261222', 'ENSG00000280230', 'ENSG00000272880', 'ENSG00000278107', 'ENSG00000248461', 'ENSG00000259839', 'ENSG00000255944', 'ENSG00000270114', 'ENSG00000265556', 'ENSG00000257477', 'ENSG00000278927', 'ENSG00000233903', 'ENSG00000244620', 'ENSG00000280141', 'ENSG00000251024', 'ENSG00000254286', 'ENSG0000

/homes/zakirov/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


### Replace unmapped gene symbols with original gene symbols

In [45]:
cur_data.adata.var['gene_symbol'] = cur_data.adata.var['gene_symbol'].fillna(
    cur_data.adata.var['gene_symbols']
)

### Validate var metadata

In [47]:
cur_data.validate_data(slot='var')

2026-04-28 16:41:18,862 INFO curation_tools.curation_tools: adata.var is valid according to the var_schema.


,ensembl_gene_id,gene_symbol
index,,
0,ENSG00000243485,MIR1302-2HG
1,ENSG00000237613,FAM138A
2,ENSG00000186092,OR4F5
3,ENSG00000241860,RP11-34P13.7
4,ENSG00000239945,RP11-34P13.8
...,...,...
33689,ENSG00000277856,AC233755.2
33690,ENSG00000275063,AC233755.1
33691,ENSG00000271254,AC240274.1


# Save the dataset

In [48]:
cur_data.save_curated_data_h5ad()

/nfs/production/mfreeberg/perturb-seq/aleks/PerturbationCatalogue/data_exploration/curation_tools/curation_tools.py:327: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  adata.obs = adata.obs.fillna(value=np.nan)


... storing 'dataset_id' as categorical
... storing 'data_modality' as categorical
... storing 'significance_criteria' as categorical
... storing 'perturbation_name' as categorical
... storing 'perturbed_target_coord' as categorical
... storing 'perturbed_target_chromosome' as categorical
... storing 'perturbed_target_ensg' as categorical
... storing 'perturbed_target_symbol' as categorical
... storing 'perturbed_target_biotype' as categorical
... storing 'guide_sequence' as categorical
... storing 'perturbation_type_label' as categorical
... storing 'perturbation_type_id' as categorical
... storing 'timepoint' as categorical
... storing 'treatment_label' as categorical
... storing 'treatment_id' as categorical
... storing 'technical_replicate' as categorical
... storing 'biological_replicate' as categorical
... storing 'model_system_label' as categorical
... storing 'model_system_id' as categorical
... storing 'species' as categorical
... storing 'tissue_label' as categorical
... stor

✅ Curated h5ad data saved to ../curated/h5ad/norman_2019_raw_curated.h5ad


In [49]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

✅ Metadata saved to ../curated/parquet/norman_2019_raw_curated_metadata.parquet


# Upload to BigQuery

In [50]:
upload_parquet_to_bq(
    parquet_path='../curated/parquet/norman_2019_raw_curated_metadata.parquet',
    bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
    bq_table_name='metadata',
    key_columns=['dataset_id', 'sample_id'],
    verbose=True
)

Staging table: loading `.parquet` file ../curated/parquet/norman_2019_raw_curated_metadata.parquet to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging...
Staging table: loaded 111445 rows to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging
Staging table: added ingested_at timestamp column to prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging
Merge completed: staging → prj-ext-dev-pertcat-437314.perturb_seq.metadata with type-safe casting.
Staging table: deleted prj-ext-dev-pertcat-437314.perturb_seq.metadata_staging


# Upload to GC Storage

In [51]:
!gcloud storage cp ../curated/h5ad/norman_2019_raw_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/

uploading large objects. If you would like to opt-out and instead
perform a normal upload, run:
`gcloud config set storage/parallel_composite_upload_enabled False`
If you would like to disable this warning, run:
`gcloud config set storage/parallel_composite_upload_enabled True`
Note that with parallel composite uploads, your object might be
uploaded as a composite object
(https://cloud.google.com/storage/docs/composite-objects), which means
that any user who downloads your object will need to use crc32c
checksums to verify data integrity. gcloud storage is capable of
computing crc32c checksums, but this might pose a problem for other
clients.

Copying file://../curated/h5ad/norman_2019_raw_curated.h5ad to gs://perturbation-catalogue-lake/perturbseq/curated/norman_2019_raw_curated.h5ad
  Completed files 32/1 | 2.7GiB/2.7GiB | 276.6MiB/s                            

Average throughput: 290.1MiB/s
